# 4 — UMAP panels per timepoint

Each merged-timepoint h5ad has its own integrated-latent UMAP (`X_umap_scvi`) containing both Dz and Lapa cells. We render:

1. Per-timepoint UMAP coloured by `scanvi_prediction`, then by `Treatment`, then by `scanvi_confidence`.
2. Rare-class highlights (Goblet, EECs, Enterocytes) per timepoint.
3. Reference (D10_Lapa) UMAP coloured by `initial_cellassign_prediction`.

Output: `figures/d2-d4-integrated/umap-panels/*.pdf`

In [ ]:
import sys, gc
from pathlib import Path

_p = Path('.').resolve()
while not (_p / 'src' / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import CELLASSIGN_DIR, FIGURES_DIR, INTEGRATION_DIR
from src.palette import celltype_palette, normalize_celltype_name

import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

TIMEPOINTS = ['D2', 'D4']
REFERENCE_H5AD = INTEGRATION_DIR / 'scarches_model' / 'reference_with_latent.h5ad'

fig_dir = FIGURES_DIR / 'd2-d4-integrated' / 'umap-panels'
fig_dir.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(fig_dir)

In [ ]:
# Per-timepoint UMAPs
for tp in TIMEPOINTS:
    a = sc.read_h5ad(CELLASSIGN_DIR / f'{tp.lower()}_scanvi_predictions.h5ad')
    if 'X_umap_scvi' in a.obsm:
        a.obsm['X_umap'] = a.obsm['X_umap_scvi']
    sc.pl.umap(a, color='scanvi_prediction', palette=celltype_palette,
               title=f'{tp}: scANVI labels (Dz + Lapa merged)',
               save=f'_{tp.lower()}_scanvi.pdf', show=True)
    sc.pl.umap(a, color='Treatment',
               title=f'{tp}: Treatment',
               save=f'_{tp.lower()}_treatment.pdf', show=True)
    sc.pl.umap(a, color='scanvi_confidence', cmap='viridis',
               title=f'{tp}: scANVI confidence',
               save=f'_{tp.lower()}_confidence.pdf', show=True)
    sc.pl.umap(a, color='dataset',
               title=f'{tp}: source dataset',
               save=f'_{tp.lower()}_dataset.pdf', show=True)
    del a; gc.collect()

In [ ]:
# Rare-class highlights per timepoint
rare_classes = ['Goblet cells', 'EECs', 'Enterocytes', 'Secretory PCs']
for tp in TIMEPOINTS:
    a = sc.read_h5ad(CELLASSIGN_DIR / f'{tp.lower()}_scanvi_predictions.h5ad')
    coords = a.obsm.get('X_umap_scvi', a.obsm.get('X_umap'))
    raw = a.obs['scanvi_prediction_raw'].astype(str)
    fig, axes = plt.subplots(1, len(rare_classes), figsize=(4*len(rare_classes), 4))
    for ax, cls in zip(axes, rare_classes):
        mask = raw == cls
        ax.scatter(coords[~mask, 0], coords[~mask, 1], s=2, c='lightgray', linewidths=0)
        ax.scatter(coords[mask, 0], coords[mask, 1], s=4,
                   c=celltype_palette.get(cls, '#000'), linewidths=0)
        ax.set_title(f'{cls} (n={int(mask.sum())})'); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'{tp}: rare-class highlights on integrated UMAP')
    plt.tight_layout()
    plt.savefig(fig_dir / f'{tp.lower()}_rare_highlights.pdf')
    plt.show()
    del a; gc.collect()

In [ ]:
# Treatment side-by-side per timepoint (same UMAP, split into two panels)
for tp in TIMEPOINTS:
    a = sc.read_h5ad(CELLASSIGN_DIR / f'{tp.lower()}_scanvi_predictions.h5ad')
    coords = a.obsm.get('X_umap_scvi', a.obsm.get('X_umap'))
    treatments = sorted(a.obs['Treatment'].astype(str).unique())
    labels = a.obs['scanvi_prediction'].astype(str).map(normalize_celltype_name)
    colors = labels.map(lambda n: celltype_palette.get(n, '#808080')).values
    fig, axes = plt.subplots(1, len(treatments), figsize=(5*len(treatments), 5), sharex=True, sharey=True)
    if len(treatments) == 1: axes = [axes]
    for ax, tx in zip(axes, treatments):
        mask = (a.obs['Treatment'].astype(str) == tx).values
        ax.scatter(coords[~mask, 0], coords[~mask, 1], s=1, c='lightgray', linewidths=0)
        ax.scatter(coords[mask, 0], coords[mask, 1], s=2, c=colors[mask], linewidths=0)
        ax.set_title(f'{tp} — {tx} (n={int(mask.sum())})'); ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(fig_dir / f'{tp.lower()}_by_treatment.pdf')
    plt.show()
    del a; gc.collect()

In [ ]:
# Reference D10 UMAP for context (its CellAssign labels)
if REFERENCE_H5AD.exists():
    ref = sc.read_h5ad(REFERENCE_H5AD)
    if 'X_umap' not in ref.obsm:
        if 'X_scANVI' in ref.obsm:
            sc.pp.neighbors(ref, use_rep='X_scANVI', n_neighbors=30)
        elif 'X_scVI' in ref.obsm:
            sc.pp.neighbors(ref, use_rep='X_scVI', n_neighbors=30)
        sc.tl.umap(ref)
    sc.pl.umap(ref, color='initial_cellassign_prediction', palette=celltype_palette,
               title='D10_Lapa reference: CellAssign labels',
               save='_d10_reference.pdf', show=True)
    del ref; gc.collect()